# MCP Runtime Observability and Continuous Assurance

Build a privacy-preserving security event, authenticate it at collection, reconstruct the operation, evaluate deterministic assurance rules, and apply only bounded authority-reducing controls. The notebook is offline and exports no telemetry.

## Learning objectives

By the end you can separate evidence from authority, inspect a safe event contract, verify signed and chained events, detect lifecycle violations, interpret detector metrics, and explain where OpenTelemetry fits in a production architecture.

## 1. Safety boundary

This lab uses synthetic support tickets, deterministic keys, an in-memory collector, and an in-memory OpenTelemetry exporter. It performs no network call and no real containment action. The model and MCP server may propose or report; the trusted host validates and authorizes; an independent observer records effects; the control plane can reduce but never grant authority.

In [ ]:
import json
import runpy
from dataclasses import asdict, replace
from datetime import timedelta
from pathlib import Path

lab = runpy.run_path(Path('lab.py'))
NOW = lab['utc']('2026-10-04T18:30:00Z')
cases = lab['scenarios']()
[(case.scenario_id, case.expected_signal, case.safe_task) for case in cases]

## 2. Inspect the security-event contract

A useful event binds correlation, reviewed artifact and contract identity, trusted pseudonymous identity, action, policy, outcome, and integrity. It omits raw customer content and credentials. A digest supports exact matching but is not automatically anonymous.

In [ ]:
safe_message = next(case for case in cases if case.scenario_id == 'safe-message')
message_events = lab['build_scenario_events'](safe_message, NOW)
request = message_events[0]
{
    'schema': request.schema_version,
    'trace_id': request.trace_id,
    'principal': request.principal_pseudonym,
    'tenant': request.tenant_pseudonym,
    'arguments': request.argument_digest,
    'destination': request.destination_digest,
    'raw_body_exported': 'Your payment is confirmed.' in json.dumps(lab['_jsonable'](asdict(request))),
}

### Privacy invariant

The emitter recursively rejects argument keys that commonly carry secrets. It permits a message body only because the complete argument object is canonicalized and hashed rather than exported. In production, use keyed/domain-separated digests for enumerable values and govern any raw evidence separately.

In [ ]:
event_json = json.dumps(lab['_jsonable'](asdict(request)), sort_keys=True)
assert 'Your payment is confirmed.' not in event_json
assert 'buyer@acme.test' not in event_json
{'event_bytes': len(event_json.encode()), 'raw_content_absent': True}

## 3. Authenticate and validate at collection

The collector checks schema, identifiers, bounded measurements, time window, emitter trust, event ID replay, canonical hash, HMAC signature, sequence, and previous-event hash. Transport encryption alone would not supply all of these evidence properties.

In [ ]:
collector = lab['SecureCollector']()
accepted = [
    collector.ingest(event, event.occurred_at + timedelta(milliseconds=4))
    for event in message_events
]
[(result.status.value, result.reason_code, result.ingestion_delay_ms) for result in accepted]

In [ ]:
tampered = replace(message_events[-1], signature='0' * 64)
tamper_collector = lab['SecureCollector']()
tamper_result = tamper_collector.ingest(
    tampered, tampered.occurred_at + timedelta(milliseconds=4)
)
{'status': tamper_result.status.value, 'reason': tamper_result.reason_code}

## 4. Reconstruct request → policy → effect → result

Correlation is useful only when the lifecycle has explicit invariants. For this single-effect operation, the verifier checks phase cardinality, order, parent links, stable context, denial/effect consistency, and final outcome. Production retries, fan-out, streaming, and compensation need a richer state machine.

In [ ]:
verifier = lab['TraceVerifier']()
valid_finding = verifier.review(message_events)
asdict(valid_finding)

In [ ]:
missing_policy = tuple(
    event for event in message_events if event.phase != lab['Phase'].POLICY
)
gap_finding = verifier.review(missing_policy)
asdict(gap_finding)

## 5. Evaluate deterministic assurance rules

Start with objective invariants: artifact or contract drift, unapproved destination, effect after deny, missing lifecycle evidence, and invalid telemetry. A repeated denial may justify review; it does not prove compromise.

In [ ]:
results = [lab['run_scenario'](case, NOW) for case in cases]
[
    {
        'scenario': result.scenario.scenario_id,
        'expected_signal': result.scenario.expected_signal,
        'detected': result.detected,
        'rule': result.signal.reason_code if result.signal else None,
        'action': result.signal.action.value if result.signal else 'none',
        'safe_task_completed': result.safe_task_completed,
    }
    for result in results
]

### Bounded control plane

The signal recommends review, quarantine, or revocation. A separate trusted control plane validates and applies the reduction. No signal can add a scope, activate a server, or approve an effect. Production controls also need authenticated requests, replay protection, expiry, operator policy, idempotency, outcome verification, and recovery.

In [ ]:
artifact_result = next(r for r in results if r.scenario.scenario_id == 'artifact-drift')
{
    'signal_action': artifact_result.signal.action.value,
    'control_result': artifact_result.control_result,
    'granted_authority': False,
}

## 6. Evaluate detection and usability together

Precision describes how often signals are correct; recall describes how many labelled unsafe cases were signalled; false-review rate measures disruption among safe cases. Safe-task completion prevents a detector that blocks everything from looking successful. Trace completeness and event rejection measure evidence health, not the same outcome as attack detection.

In [ ]:
campaign = lab['run_campaign'](cases, NOW)
metrics = campaign.metrics()
{
    'confusion': asdict(campaign.confusion),
    'metrics': metrics,
    'recall_95pct_wilson': lab['wilson_interval'](
        campaign.confusion.true_positive,
        campaign.confusion.true_positive + campaign.confusion.false_negative,
    ),
}

In [ ]:
[asdict(item) for item in lab['evaluate_slos'](metrics)]

## 7. Inspect real OpenTelemetry context

The probe creates host, MCP call, and backend spans with the OpenTelemetry Python SDK and an in-memory exporter. It checks that all spans share one trace, parent relationships are valid, a W3C `traceparent` exists, and only a bounded attribute allowlist is used. This proves instrumentation behavior, not collector or network security.

In [ ]:
otel = lab['run_otel_probe']()
asdict(otel)

## 8. Production design exercise

For one consequential tool in your architecture, write down: the trusted identity source; reviewed artifact and contract identifiers; event fields and excluded data; independent effect oracle; mandatory unsampled events; collector authentication and redaction; deterministic rules; labelled evaluation set; bounded actions; recovery path; retention and access policy; and pipeline-loss behavior. Then test one unknown-outcome case rather than treating timeout as success or denial.

## Reflection

1. Which event fields came from trusted application state and which could an MCP peer influence?
2. Why does a valid event signature not prove that every real effect was reported?
3. Which evidence in your system must survive probabilistic sampling?
4. When should missing telemetry cause review, quarantine, or fail-closed behavior?
5. How would you detect that the collector or assurance pipeline itself is failing?
6. What metric prevents an over-broad detector from appearing safe?
7. What additional controls are needed before an automated signal can revoke production authority?